=============================================================================
Day 1A — De Prompt a Acción: Tu Primer Agente con LangGraph
=============================================================================

Estructura del grafo ReAct que construiremos:

  [START]

     │
     ▼
  
  [agent]  ◄──────────────────┐
     
     │                        │
     ├── tool_calls ──► [tools]
     │                        │
     └── stop ──► [END]       └───┘


Cada nodo es una función Python. El estado es un dict tipado con MessagesState
que fluye entre nodos. LangGraph hace explícito el loop que antes escribíamos
a mano con un `while`.

=============================================================================

## Sección 1: Instalación

Instala las dependencias necesarias para el curso.

## Sección 2: Configuración del Backend

Elige tu proveedor de LLM cambiando `BACKEND`.

| Backend | Modelo | Requiere |
|---------|--------|----------|
| `ollama` | qwen2.5:7b | Ollama local |
| `anthropic` | claude-3-5-haiku | `ANTHROPIC_API_KEY` |
| `openai` | gpt-4o-mini | `OPENAI_API_KEY` |
| `groq` | llama-3.3-70b | `GROQ_API_KEY` |
| `nvidia` | meta/llama-3.3-70b-instruct | `NVIDIA_API_KEY` |

In [1]:
import os
import httpx
import urllib.parse
from ddgs import DDGS
from typing import Literal
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode

# ════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — cambia BACKEND para elegir tu proveedor
# ════════════════════════════════════════════════════════════
load_dotenv()

BACKEND = "nvidia"   # opciones: "ollama" | "anthropic" | "openai" | "groq" | "nvidia"

CONFIGS = {
    # Ollama corriendo localmente (gratis, sin API key)
    # Instalar: https://ollama.com  →  ollama pull qwen2.5:7b
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key":  "ollama",
        "model":    "qwen2.5:7b",        # alternativas: llama3.1, mistral-nemo
    },
    # Anthropic vía endpoint OpenAI-compatible
    # Requiere: export ANTHROPIC_API_KEY=sk-ant-...
    "anthropic": {
        "base_url": "https://api.anthropic.com/v1",
        "api_key":  os.getenv("ANTHROPIC_API_KEY", ""),
        "model":    "claude-3-5-haiku-20241022",
    },
    # OpenAI
    # Requiere: export OPENAI_API_KEY=sk-...
    "openai": {
        "base_url": None,
        "api_key":  os.getenv("OPENAI_API_KEY", ""),
        "model":    "gpt-4o-mini",
    },
    # Groq — tier gratuito disponible: https://console.groq.com
    # Requiere: export GROQ_API_KEY=gsk_...
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "api_key":  os.getenv("GROQ_API_KEY", ""),
        "model":    "llama-3.3-70b-versatile",
    },
    # NVIDIA NIM (catálogo de 50+ modelos, tier gratuito con 1,000 créditos)
    # Requiere: NVIDIA_API_KEY en entorno  →  https://build.nvidia.com
    # El API key empieza con nvapi-
    # Modelos con tool use: meta/llama-3.3-70b-instruct, mistralai/mistral-large-latest
    "nvidia": {
        "base_url": "https://integrate.api.nvidia.com/v1",
        "api_key":  os.getenv("NVIDIA_API_KEY", ""),
        "model":    "meta/llama-3.3-70b-instruct",  # o cualquier otro del catálogo
    },
}

cfg = CONFIGS[BACKEND]
llm_kwargs = {"model": cfg["model"], "api_key": cfg["api_key"]}
if cfg["base_url"]:
    llm_kwargs["base_url"] = cfg["base_url"]

llm = ChatOpenAI(**llm_kwargs)
print(f"✅ Backend: {BACKEND.upper()} | Modelo: {cfg['model']}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


## 🔨 Sección 3: Herramientas con `@tool`

El decorador `@tool` extrae automáticamente:
- **nombre** → nombre de la función
- **descripción** → docstring
- **schema JSON** → type hints de los parámetros

No hace falta escribir el JSON schema a mano.

In [2]:
@tool
def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        output = []
        for r in results:
            output.append(f"**{r['title']}**\n{r['body']}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error al buscar: {e}"


@tool
def get_weather(city: str) -> str:
    """Obtiene el clima actual de una ciudad específica."""
    try:
        r = httpx.get(f"https://wttr.in/{urllib.parse.quote(city)}?format=3", timeout=8)
        return r.text.strip()
    except Exception as e:
        return f"Error: {e}"


@tool
def calculate(expression: str) -> str:
    """Evalúa una expresión matemática simple, ej: '(15 * 8) / 3 + 7'."""
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return "Error: solo expresiones numéricas básicas."
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


TOOLS = [web_search, get_weather, calculate]

print(f"✅ {len(TOOLS)} herramientas definidas: {[t.name for t in TOOLS]}")

✅ 3 herramientas definidas: ['web_search', 'get_weather', 'calculate']


## 🔄 Sección 4: Construir el Grafo ReAct

```
[START]
   │
   ▼
[agent]  ◄──────────────────┐
   │                        │
   ├── tool_calls ──► [tools]
   │                        │
   └── stop ──► [END]       └───┘
```

- **`agent_node`** — llama al LLM con el estado actual
- **`tool_node`** — ejecuta la tool y agrega el resultado al estado
- **`router`** — edge condicional: `tool_calls` → tools | `stop` → END

In [3]:
llm_with_tools = llm.bind_tools(TOOLS)

SYSTEM_PROMPT = (
    "Eres un asistente útil con acceso a herramientas. "
    "Usa las herramientas disponibles cuando necesites información actualizada "
    "o realizar cálculos. Responde en el idioma del usuario."
)


def agent_node(state: MessagesState) -> dict:
    """
    Nodo 'agent': recibe el estado con los mensajes hasta ahora,
    llama al LLM y devuelve su respuesta (puede incluir tool_calls).
    """
    # Prepend system message si no existe
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages

    response = llm_with_tools.invoke(messages)
    # LangGraph mergea automáticamente la lista devuelta al estado
    return {"messages": [response]}


def router(state: MessagesState) -> Literal["tools", END]:
    """
    Edge condicional: inspecciona el último mensaje del asistente.
    Si tiene tool_calls → ir al nodo 'tools'.
    Si no              → terminar (END).
    """
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END


# ToolNode maneja el dispatch automático: recibe el tool_call del LLM,
# ejecuta la función Python correspondiente y agrega el ToolMessage al estado.
tool_node = ToolNode(TOOLS)

# ── Ensamblar el grafo ────────────────────────────────────────────────────────
builder = StateGraph(MessagesState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", router)   # router decide a dónde ir
builder.add_edge("tools", "agent")               # después de tools, vuelve a agent

graph = builder.compile()

print("✅ Grafo compilado.")
print("   Nodos:", list(graph.nodes))

✅ Grafo compilado.
   Nodos: ['__start__', 'agent', 'tools']


## Sección 5: Visualizar el Grafo (opcional)

Requiere `pip install grandalf`. Si no está disponible se omite silenciosamente.

In [4]:
try:
    graph_ascii = graph.get_graph().draw_ascii()
    print("\nEstructura del grafo:")
    print(graph_ascii)
except Exception:
    print("(Instala grandalf para visualización ASCII del grafo)")


Estructura del grafo:
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## Sección 6: Función Helper `run_agent`

Helper que invoca el grafo con `stream_mode="values"` para ver
el estado completo después de cada nodo.

In [5]:
def run_agent(user_message: str, verbose: bool = True) -> str:
    """
    Invoca el grafo con un mensaje de usuario y retorna la respuesta final.

    El grafo ejecuta el loop completo internamente:
    agent → [tools → agent]* → END
    """
    if verbose:
        print(f"\n{'─'*60}")
        print(f"👤 Usuario: {user_message}")

    # stream_mode="values" entrega el estado completo después de cada nodo
    final_state = None
    for step in graph.stream(
        {"messages": [HumanMessage(content=user_message)]},
        stream_mode="values",
    ):
        final_state = step
        if verbose:
            last_msg = step["messages"][-1]
            label = last_msg.__class__.__name__
            # Mostrar tool calls si existen
            if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"  🔧 [{label}] {tc['name']}({tc['args']})")
            elif hasattr(last_msg, "name") and last_msg.name:
                # ToolMessage con resultado
                preview = str(last_msg.content)[:150]
                print(f"  📥 [ToolResult:{last_msg.name}] {preview}")

    final_text = final_state["messages"][-1].content
    if verbose:
        print(f"\n🤖 Respuesta:\n{final_text}")
    return final_text

## Sección 7: Demos

Prueba el agente con diferentes tipos de consultas:

1. Pregunta sin herramientas
2. Búsqueda web
3. Clima
4. Múltiples herramientas encadenadas
5. Inspección paso a paso del estado

In [6]:
print("\n" + "=" * 60)
print("DEMO 1: Pregunta sin herramientas")
print("=" * 60)
run_agent("¿Cuántas lunas tiene Marte?")


DEMO 1: Pregunta sin herramientas

────────────────────────────────────────────────────────────
👤 Usuario: ¿Cuántas lunas tiene Marte?
  🔧 [AIMessage] web_search({'query': '¿Cuántas lunas tiene Marte?'})
  📥 [ToolResult:web_search] **¿Cuántas lunas tiene cada planeta? - NASA Space Place**
Apr 13, 2026 · Marte tiene dos lunas, que se llaman Fobos y Deimos. Esta página de NASA expl

🤖 Respuesta:
Marte tiene dos lunas, que se llaman Fobos y Deimos.


'Marte tiene dos lunas, que se llaman Fobos y Deimos.'

In [7]:
print("\n" + "=" * 60)
print("DEMO 2: Búsqueda web")
print("=" * 60)
run_agent("¿Qué es el Agent Development Kit de Google?")


DEMO 2: Búsqueda web

────────────────────────────────────────────────────────────
👤 Usuario: ¿Qué es el Agent Development Kit de Google?
  🔧 [AIMessage] web_search({'query': 'Agent Development Kit de Google'})
  📥 [ToolResult:web_search] **Agent Development Kit | Gemini Enterprise Agent Platform ...**
3 days ago · Agent Development Kit (ADK) is an open-source agent development framewor

🤖 Respuesta:
El Agent Development Kit (ADK) de Google es un marco de desarrollo de agentes de código abierto que permite a los desarrolladores crear, depurar y implementar agentes de inteligencia artificial confiables a escala empresarial. Está disponible en Python, TypeScript, Go y Java, y es el mismo marco que impulsa a los agentes dentro de los productos de Google como Agentspace y el Google Customer Engagement Suite (CES). El ADK permite a los desarrolladores crear aplicaciones de agentes listas para producción con mayor flexibilidad y control preciso.


'El Agent Development Kit (ADK) de Google es un marco de desarrollo de agentes de código abierto que permite a los desarrolladores crear, depurar y implementar agentes de inteligencia artificial confiables a escala empresarial. Está disponible en Python, TypeScript, Go y Java, y es el mismo marco que impulsa a los agentes dentro de los productos de Google como Agentspace y el Google Customer Engagement Suite (CES). El ADK permite a los desarrolladores crear aplicaciones de agentes listas para producción con mayor flexibilidad y control preciso.'

In [8]:
print("\n" + "=" * 60)
print("DEMO 3: Clima")
print("=" * 60)
run_agent("¿Cómo está el clima en Medellín ahora?")


DEMO 3: Clima

────────────────────────────────────────────────────────────
👤 Usuario: ¿Cómo está el clima en Medellín ahora?
  🔧 [AIMessage] get_weather({'city': 'Medellín'})
  📥 [ToolResult:get_weather] medellín: ⛅  +72°F

🤖 Respuesta:
El clima en Medellín ahora es de 72°F con cielos parcialmente nublados.


'El clima en Medellín ahora es de 72°F con cielos parcialmente nublados.'

In [9]:
print("\n" + "=" * 60)
print("DEMO 4: Múltiples herramientas encadenadas")
print("=" * 60)
run_agent(
    "Busca el precio actual de Bitcoin en USD "
    "y calcula cuánto costarían 0.75 BTC."
)


DEMO 4: Múltiples herramientas encadenadas

────────────────────────────────────────────────────────────
👤 Usuario: Busca el precio actual de Bitcoin en USD y calcula cuánto costarían 0.75 BTC.
  🔧 [AIMessage] web_search({'query': 'precio actual de Bitcoin en USD'})
  📥 [ToolResult:web_search] **BTC a USD: intercambiar Bitcoin (BTC) a Dólar estadounidense (USD)**
Intercambiar Bitcoin BTC a United States Dollar USD. BTC a USD: 1 Bitcoin se co
  🔧 [AIMessage] calculate({'expression': '76978.9 * 0.75'})
  📥 [ToolResult:calculate] 57734.174999999996

🤖 Respuesta:
El precio actual de Bitcoin en USD es de $76,260.00 por BTC. 
0.75 BTC costarían $57,734.17.


'El precio actual de Bitcoin en USD es de $76,260.00 por BTC. \n0.75 BTC costarían $57,734.17.'

In [10]:
print("\n" + "=" * 60)
print("DEMO 5: Inspección paso a paso del estado")
print("=" * 60)
print("(Muestra el estado completo después de cada nodo del grafo)\n")
for i, step in enumerate(graph.stream(
    {"messages": [HumanMessage(content="¿Cuál es el clima en Bogotá?")]},
    stream_mode="values",
)):
    print(f"[Step {i}]")
    for msg in step["messages"]:
        cls = msg.__class__.__name__
        content_preview = str(msg.content)[:100] if msg.content else ""
        tc_info = ""
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            tc_info = f" | tool_calls={[tc['name'] for tc in msg.tool_calls]}"
        print(f"  {cls}{tc_info}: {content_preview}")
    print()


DEMO 5: Inspección paso a paso del estado
(Muestra el estado completo después de cada nodo del grafo)

[Step 0]
  HumanMessage: ¿Cuál es el clima en Bogotá?

[Step 1]
  HumanMessage: ¿Cuál es el clima en Bogotá?
  AIMessage | tool_calls=['get_weather']: 

[Step 2]
  HumanMessage: ¿Cuál es el clima en Bogotá?
  AIMessage | tool_calls=['get_weather']: 
  ToolMessage: bogotá: ⛅  +61°F

[Step 3]
  HumanMessage: ¿Cuál es el clima en Bogotá?
  AIMessage | tool_calls=['get_weather']: 
  ToolMessage: bogotá: ⛅  +61°F
  AIMessage: El clima en Bogotá es de 61°F con cielos parcialmente nublados.

